# 01 - Pose score calibration (T2, issue #3)

Posture score v1: ear-shoulder verticality plus head pitch plus shoulder
symmetry plus an experimental shoulder-posture proxy, 0-100, higher means
more upright. Inputs are numpy `(17, 2)` normalized keypoints plus `(17,)`
visibility (COCO-17: nose 0, eyes 1-2, ears 3-4, shoulders 5-6, hips 11-12
— confirmed live on `yolo26n-pose.pt`). Torch conversion happens at the call
edge, never inside the score function. Indeterminate frames yield `None`
(hold last state; T3 owns dwell timing). No persistence (ADR-0002).

Calibrated 2026-09-25 (synthetic desk fixtures; live tuning is an open knob):
- weights: neck 0.5, head-pitch 0.25, symmetry 0.15, shoulder-posture 0.10
- visibility floor 0.5 (below -> keypoint ignored)
- pitch: healthy deadzone 0.25, fully-bowed reference 0.9
- posture proxy: upright width ratio 0.5, hunched reference 0.3
- valid range upright at score >= 70, recovery hysteresis at score >= 80
Geometry runs in aspect-corrected space: pass frame W/H as aspect
(T3 supplies it from the frame shape); default 1.0 keeps fixture numbers
in true-proportion space.

Known frontal-2D limits (not solved, documented):
- no left/right gaze (yaw): lateral ear displacement conflates lean, tilt, turn
- shoulder-posture proxy is EXPERIMENTAL: confounded by clothing, body shape,
  sitting vs standing; requires live calibration before trusting its weight

In [ ]:
import math

In [ ]:
EAR_L, EAR_R, SHOULDER_L, SHOULDER_R = 3, 4, 5, 6
NOSE, EYE_L, EYE_R = 0, 1, 2
HIP_L, HIP_R = 11, 12
VISIBILITY_FLOOR = 0.5
NECK_WEIGHT = 0.5
PITCH_WEIGHT = 0.25
SYMMETRY_WEIGHT = 0.15
POSTURE_WEIGHT = 0.10
PITCH_OK_RATIO = 0.25
PITCH_REF_RATIO = 0.9
UPRIGHT_WIDTH_RATIO = 0.5
HUNCHED_WIDTH_RATIO = 0.3


def _midpoint(left, right):
    return ((left[0] + right[0]) / 2.0, (left[1] + right[1]) / 2.0)


def _side_verticality(ear, shoulder, vis_ear, vis_shoulder):
    """Neck angle from vertical as cosine; None when the side is unseen."""
    if min(vis_ear, vis_shoulder) < VISIBILITY_FLOOR:
        return None
    dx = abs(ear[0] - shoulder[0])
    dy = shoulder[1] - ear[1]
    if dy <= 0:
        return 0.0
    return math.cos(math.atan2(dx, dy))


def _shoulder_symmetry(shoulder_l, shoulder_r, vis_l, vis_r):
    """Shoulder levelness; None when shoulders are unseen."""
    if min(vis_l, vis_r) < VISIBILITY_FLOOR:
        return None
    width = abs(shoulder_l[0] - shoulder_r[0])
    level_gap = abs(shoulder_l[1] - shoulder_r[1])
    if width <= 0:
        return 1.0 if level_gap == 0 else 0.0
    return max(0.0, 1.0 - level_gap / width)


def head_pitch(xyn, visibility):
    """Level gaze as 0-1 (1 = level); None when nose/eyes/shoulders unseen."""
    needed = (NOSE, EYE_L, EYE_R, SHOULDER_L, SHOULDER_R)
    if min(visibility[i] for i in needed) < VISIBILITY_FLOOR:
        return None
    eyes_mid = _midpoint(xyn[EYE_L], xyn[EYE_R])
    shoulders_mid = _midpoint(xyn[SHOULDER_L], xyn[SHOULDER_R])
    head_size = shoulders_mid[1] - eyes_mid[1]
    if head_size <= 0:
        return None
    ratio = (xyn[NOSE][1] - eyes_mid[1]) / head_size
    if ratio <= PITCH_OK_RATIO:
        return 1.0
    if ratio >= PITCH_REF_RATIO:
        return 0.0
    return 1.0 - (ratio - PITCH_OK_RATIO) / (PITCH_REF_RATIO - PITCH_OK_RATIO)


def shoulder_posture(xyn, visibility):
    """EXPERIMENTAL shoulder width vs torso 0-1; None when unseen."""
    needed = (SHOULDER_L, SHOULDER_R, HIP_L, HIP_R)
    if min(visibility[i] for i in needed) < VISIBILITY_FLOOR:
        return None
    shoulders_mid = _midpoint(xyn[SHOULDER_L], xyn[SHOULDER_R])
    hips_mid = _midpoint(xyn[HIP_L], xyn[HIP_R])
    torso = hips_mid[1] - shoulders_mid[1]
    if torso <= 0:
        return None
    width = abs(xyn[SHOULDER_L][0] - xyn[SHOULDER_R][0])
    ratio = width / torso
    if ratio >= UPRIGHT_WIDTH_RATIO:
        return 1.0
    if ratio <= HUNCHED_WIDTH_RATIO:
        return 0.0
    return (ratio - HUNCHED_WIDTH_RATIO) / (UPRIGHT_WIDTH_RATIO - HUNCHED_WIDTH_RATIO)


def posture_score(xyn, visibility, aspect=1.0):
    """Score 0-100 from available evidence; None when indeterminate.

    x is scaled once into aspect-corrected space (x * W/H) because
    normalized x/y units differ on non-square frames.
    """
    if aspect <= 0:
        raise ValueError("aspect must be positive")
    corrected = [[point[0] * aspect, point[1]] for point in xyn]
    sides = []
    for ear_i, shoulder_i in ((EAR_L, SHOULDER_L), (EAR_R, SHOULDER_R)):
        value = _side_verticality(
            corrected[ear_i], corrected[shoulder_i], visibility[ear_i], visibility[shoulder_i]
        )
        if value is not None:
            sides.append(value)
    parts = []
    if sides:
        parts.append((sum(sides) / len(sides), NECK_WEIGHT))
    pitch = head_pitch(corrected, visibility)
    if pitch is not None:
        parts.append((pitch, PITCH_WEIGHT))
    symmetry = _shoulder_symmetry(
        corrected[SHOULDER_L], corrected[SHOULDER_R], visibility[SHOULDER_L], visibility[SHOULDER_R]
    )
    if symmetry is not None:
        parts.append((symmetry, SYMMETRY_WEIGHT))
    posture = shoulder_posture(corrected, visibility)
    if posture is not None:
        parts.append((posture, POSTURE_WEIGHT))
    if not parts:
        return None
    total = sum(weight for _, weight in parts)
    return 100.0 * sum(value * weight for value, weight in parts) / total

In [ ]:
UPRIGHT_SCORE = 70
RECOVERED_SCORE = 80


def select_largest_person(persons):
    """Largest box area wins; None when nobody is visible.

    persons: list of {"area", "xyn", "visibility"} dicts. Consumed by T3."""
    if not persons:
        return None
    return max(persons, key=lambda person: person["area"])


def is_upright(score):
    """Valid range: at or above 70. None is never upright. Consumed by T3."""
    return bool(score is not None and score >= UPRIGHT_SCORE)


def is_recovered(score):
    """Recovery hysteresis: at or above 80. None never recovers. Consumed by T3."""
    return bool(score is not None and score >= RECOVERED_SCORE)